In [1]:
import sqlite3

# 1. Establish connection (ye line automatically 'supply_chain.db' file create kar degi agar wo exist nahi karti)
conn = sqlite3.connect('supply_chain.db')
cursor = conn.cursor()

# 2. Create the Products Dimension Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    unit_cost REAL NOT NULL
)
''')

# 3. Create the Warehouses Dimension Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS warehouses (
    warehouse_id INTEGER PRIMARY KEY,
    location TEXT NOT NULL,
    capacity INTEGER NOT NULL
)
''')

# 4. Save the changes
conn.commit()

print("Database aur dimension tables successfully set up ho gayi hain!")

Database aur dimension tables successfully set up ho gayi hain!


In [2]:
# 1. Create the Fact Table: sales_transactions
cursor.execute('''
CREATE TABLE IF NOT EXISTS sales_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER,
    date TEXT NOT NULL,
    quantity_sold INTEGER NOT NULL,
    price REAL NOT NULL,
    FOREIGN KEY (product_id) REFERENCES products (product_id) ON DELETE CASCADE
)
''')

# 2. Create the Analytical Table: inventory_predictions
# (Yahan hum baad me apne ML model ke outputs write karenge)
cursor.execute('''
CREATE TABLE IF NOT EXISTS inventory_predictions (
    prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER,
    forecast_date TEXT NOT NULL,
    predicted_demand REAL,
    reorder_point REAL,
    safety_stock REAL,
    FOREIGN KEY (product_id) REFERENCES products (product_id) ON DELETE CASCADE
)
''')

conn.commit()
print("Fact aur Analytical tables successfully create ho gayi!")

Fact aur Analytical tables successfully create ho gayi!


In [3]:
import random
from datetime import datetime, timedelta

# 1. Insert Dummy Products
cursor.executemany('''
INSERT INTO products (product_name, category, unit_cost) 
VALUES (?, ?, ?)
''', [
    ('Textured Polo', 'Apparel', 450.0), 
    ('Relaxed Fit Denim', 'Apparel', 800.0)
])

# 2. Generate 30 days of dummy sales data
start_date = datetime.now() - timedelta(days=30)
sales_data = []

for i in range(30):
    current_date = (start_date + timedelta(days=i)).strftime('%Y-%m-%d')
    
    # Random daily sales for Product 1 (Textured Polo)
    sales_data.append((1, current_date, random.randint(10, 50), 999.0))
    
    # Random daily sales for Product 2 (Relaxed Fit Denim)
    sales_data.append((2, current_date, random.randint(5, 30), 1499.0))

# 3. Insert Sales Data into Fact Table
cursor.executemany('''
INSERT INTO sales_transactions (product_id, date, quantity_sold, price) 
VALUES (?, ?, ?, ?)
''', sales_data)

conn.commit()
print("Database is now populated with products and 30 days of historical sales data.")

Database is now populated with products and 30 days of historical sales data.


In [4]:
import pandas as pd

# Advanced SQL Query with CTEs and Window Functions
sql_query = """
WITH DailySales AS (
    -- Step A: Pehle hum daily total quantity nikal rahe hain har product ke liye
    SELECT 
        product_id,
        date,
        SUM(quantity_sold) as daily_demand
    FROM sales_transactions
    GROUP BY product_id, date
)
SELECT 
    product_id,
    date,
    daily_demand,
    
    -- Step B: 7-Day Rolling Average (Pichle 7 din ki average demand)
    AVG(daily_demand) OVER(
        PARTITION BY product_id 
        ORDER BY date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) as rolling_7d_avg,
    
    -- Step C: Lag of 1 Day (Kal kitni sale hui thi?)
    LAG(daily_demand, 1) OVER(
        PARTITION BY product_id 
        ORDER BY date
    ) as lag_1d_demand,
    
    -- Step D: Lag of 7 Days (Pichle hafte same din kitni sale hui thi?)
    LAG(daily_demand, 7) OVER(
        PARTITION BY product_id 
        ORDER BY date
    ) as lag_7d_demand

FROM DailySales;
"""

# Fetching the optimized dataset directly into a Pandas DataFrame
df = pd.read_sql_query(sql_query, conn)

# Displaying the first 15 rows to verify our engineered features
print(df.head(15))

    product_id        date  daily_demand  rolling_7d_avg  lag_1d_demand  \
0            1  2026-04-27            20       20.000000            NaN   
1            1  2026-04-28            29       24.500000           20.0   
2            1  2026-04-29            34       27.666667           29.0   
3            1  2026-04-30            13       24.000000           34.0   
4            1  2026-05-01            33       25.800000           13.0   
5            1  2026-05-02            25       25.666667           33.0   
6            1  2026-05-03            40       27.714286           25.0   
7            1  2026-05-04            43       31.000000           40.0   
8            1  2026-05-05            18       29.428571           43.0   
9            1  2026-05-06            36       29.714286           18.0   
10           1  2026-05-07            22       31.000000           36.0   
11           1  2026-05-08            23       29.571429           22.0   
12           1  2026-05-0

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# 1. Handle Missing Values: Drop rows with NaN (initial days without enough history)
df_clean = df.dropna().copy()

# Sort data chronologically to prevent future data from leaking into past training data
df_clean = df_clean.sort_values(by=['product_id', 'date'])

# 2. Define Features (X) and Target (y)
features = ['rolling_7d_avg', 'lag_1d_demand', 'lag_7d_demand']
X = df_clean[features]
y = df_clean['daily_demand']

# 3. Train-Test Split (Chronological for Time Series)
# Hum shuruati 80% din model ko train karne ke liye denge, aur last 20% din par usko test karenge
split_index = int(len(df_clean) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 4. Initialize and Train the Random Forest Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Make Predictions and Evaluate
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print("Model Training Successfully Completed!\n")
print(f"Mean Absolute Error (MAE): {mae:.2f} units (Iska matlab humari prediction average itne units se off ho sakti hai)")

# 6. Visualize Predictions vs Actual in a DataFrame
results_df = df_clean.iloc[split_index:].copy()
results_df['predicted_demand'] = np.round(predictions, 1)

print("\nLast 5 Days Predictions vs Actual Demand:")
print(results_df[['product_id', 'date', 'daily_demand', 'predicted_demand']].tail())

Model Training Successfully Completed!

Mean Absolute Error (MAE): 10.88 units (Iska matlab humari prediction average itne units se off ho sakti hai)

Last 5 Days Predictions vs Actual Demand:
    product_id        date  daily_demand  predicted_demand
55           2  2026-05-22            12              21.5
56           2  2026-05-23             6              20.9
57           2  2026-05-24            23              10.3
58           2  2026-05-25            18              22.0
59           2  2026-05-26            24              11.5


In [6]:
import scipy.stats as st
import numpy as np

# 1. Define Variables
LEAD_TIME_DAYS = 5 # Assume karte hain supplier 5 din me inventory deliver karta hai
# Z-score for 95% service level (Yani 95% certainty ki stockout nahi hoga)
Z_SCORE_95 = st.norm.ppf(0.95) 

# 2. Calculate Standard Deviation of Historical Demand
# Har product ki variability alag hoti hai, isliye hum product-level standard deviation nikalenge
std_dev_demand = df_clean.groupby('product_id')['daily_demand'].std().reset_index()
std_dev_demand.rename(columns={'daily_demand': 'std_dev'}, inplace=True)

# Merge the standard deviation back to our results dataframe
results_df = results_df.merge(std_dev_demand, on='product_id', how='left')

# 3. Calculate Safety Stock
# Formula: Z * Standard Deviation * sqrt(Lead Time)
results_df['safety_stock'] = np.round(
    Z_SCORE_95 * results_df['std_dev'] * np.sqrt(LEAD_TIME_DAYS), 0
)

# 4. Calculate Dynamic Reorder Point
# Formula: (Predicted Demand * Lead Time) + Safety Stock
results_df['reorder_point'] = np.round(
    (results_df['predicted_demand'] * LEAD_TIME_DAYS) + results_df['safety_stock'], 0
)

print("Step 4: Inventory Optimization Metrics Calculated Successfully!\n")
print(results_df[['product_id', 'date', 'predicted_demand', 'safety_stock', 'reorder_point']].tail())

Step 4: Inventory Optimization Metrics Calculated Successfully!

   product_id        date  predicted_demand  safety_stock  reorder_point
5           2  2026-05-22              21.5          28.0          136.0
6           2  2026-05-23              20.9          28.0          132.0
7           2  2026-05-24              10.3          28.0           80.0
8           2  2026-05-25              22.0          28.0          138.0
9           2  2026-05-26              11.5          28.0           86.0


In [7]:
# 1. Select and rename columns to match our database schema
final_db_data = results_df[['product_id', 'date', 'predicted_demand', 'reorder_point', 'safety_stock']].copy()
final_db_data.rename(columns={'date': 'forecast_date'}, inplace=True)

# 2. Write the DataFrame directly to the SQLite database
# if_exists='append' ensures we add new predictions without deleting historical ones
final_db_data.to_sql('inventory_predictions', conn, if_exists='append', index=False)

print("Pipeline Complete: ML predictions aur inventory metrics successfully database me write ho gaye hain!\n")

# 3. Verify the data was successfully inserted
cursor.execute("SELECT * FROM inventory_predictions LIMIT 5")
print("Database Verification (First 5 inserted rows):")
for row in cursor.fetchall():
    print(row)

# 4. Close the database connection (Crucial best practice!)
conn.close()

Pipeline Complete: ML predictions aur inventory metrics successfully database me write ho gaye hain!

Database Verification (First 5 inserted rows):
(1, 2, '2026-05-17', 10.3, 80.0, 28.0)
(2, 2, '2026-05-18', 22.6, 141.0, 28.0)
(3, 2, '2026-05-19', 18.5, 120.0, 28.0)
(4, 2, '2026-05-20', 20.8, 132.0, 28.0)
(5, 2, '2026-05-21', 21.5, 136.0, 28.0)
